In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Loading CSV file
df = pd.read_csv('Mortage.csv')

# Text and labels
X = df['text'].values  
y = df['label'].values

# Encode the labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [2]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Tokenize the text data
tokenizer = Tokenizer(num_words=5000)  # Use top 5000 words
tokenizer.fit_on_texts(X_train)

In [3]:
# Convert text to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences to ensure all are the same length
maxlen = 100 
X_train_pad = pad_sequences(X_train_seq, padding='post', maxlen=maxlen)
X_test_pad = pad_sequences(X_test_seq, padding='post', maxlen=maxlen)


In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense, Dropout

# Building CNN model
vocab_size = 5000
embedding_dim = 100

model = Sequential()

# Embedding layer
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen))

# Convolutional layer
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))

# Max-pooling layer
model.add(MaxPooling1D(pool_size=2))

# Global Max-pooling to reduce the dimensionality further
model.add(GlobalMaxPooling1D())

# Dropout layer for regularization
model.add(Dropout(0.5))

# Fully connected dense layer
model.add(Dense(10, activation='relu'))

# Output layer
model.add(Dense(3, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# model summary
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 100)          500000    
                                                                 
 conv1d (Conv1D)             (None, 96, 128)           64128     
                                                                 
 max_pooling1d (MaxPooling1  (None, 48, 128)           0         
 D)                                                              
                                                                 
 global_max_pooling1d (Glob  (None, 128)               0         
 alMaxPooling1D)                                                 
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense (Dense)               (None, 10)                1

In [6]:
# Train the model
history = model.fit(X_train_pad, y_train, epochs=10, batch_size=64, validation_split=0.2)

# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test_pad, y_test)
print(f'Test Accuracy: {accuracy}')

Epoch 1/10
1/1 [==============================] - 0s 130ms/step - loss: 0.9517 - accuracy: 0.6667 - val_loss: 1.0911 - val_accuracy: 0.2500
Epoch 2/10
1/1 [==============================] - 0s 76ms/step - loss: 0.9414 - accuracy: 0.6667 - val_loss: 1.0889 - val_accuracy: 0.2500
Epoch 3/10
1/1 [==============================] - 0s 73ms/step - loss: 0.9258 - accuracy: 0.9167 - val_loss: 1.0867 - val_accuracy: 0.2500
Epoch 4/10
1/1 [==============================] - 0s 81ms/step - loss: 0.9055 - accuracy: 0.7500 - val_loss: 1.0839 - val_accuracy: 0.2500
Epoch 5/10
1/1 [==============================] - 0s 69ms/step - loss: 0.8818 - accuracy: 0.8333 - val_loss: 1.0808 - val_accuracy: 0.2500
Epoch 6/10
1/1 [==============================] - 0s 67ms/step - loss: 0.8604 - accuracy: 0.7500 - val_loss: 1.0782 - val_accuracy: 0.2500
Epoch 7/10
1/1 [==============================] - 0s 61ms/step - loss: 0.7991 - accuracy: 0.8333 - val_loss: 1.0756 - val_accuracy: 0.2500
Epoch 8/10
1/1 [==========

In [7]:
# predictions on the test data
y_pred = model.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred, axis=1)

# Converting numeric predictions back to label names
y_pred_labels = label_encoder.inverse_transform(y_pred_classes)

# Comparing predictions with the true labels
print(y_pred_labels[:10])  
print(y_test[:10])


1/1 [==============================] - 0s 165ms/step
['Mortage' 'Trust' 'Mortage' 'Mortage' 'Mortage']
[0 2 2 0 1]
